In [ ]:
pip install opencv-python numpy matplotlib

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import cv2
import numpy as np
import os

class ImageQualityChecker:

    def blur_score(self, image):

        if image is None:
            return 0

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        return cv2.Laplacian(gray, cv2.CV_64F).var()

    def noise_score(self, image):

        if image is None:
            return 0

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray,(5,5),0)
        return np.mean(np.abs(gray - blur))

    def geo_score(self, image):

        if image is None:
            return 0

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        edges = cv2.Canny(gray,50,150)

        lines = cv2.HoughLines(edges,1,np.pi/180,120)

        if lines is None:
            return 100

        angles = []

        for line in lines[:20]:
            rho, theta = line[0]
            angle = theta * 180 / np.pi
            angles.append(angle)

        return np.std(angles)

    # Alignment check for transform
    def alignment_error(self, image, reference):

        if image is None or reference is None:
            return 100

        orb = cv2.ORB_create(800)

        kp1, des1 = orb.detectAndCompute(reference, None)
        kp2, des2 = orb.detectAndCompute(image, None)

        if des1 is None or des2 is None:
            return 100

        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

        matches = bf.match(des1, des2)

        if len(matches) < 10:
            return 100

        distances = [m.distance for m in matches]

        return np.mean(distances)


checker = ImageQualityChecker()


# -----------------------------
# Reference Images
# -----------------------------

good_example = r"/content/drive/MyDrive/tiles/tile_03.png"
blur_example = r"/content/drive/MyDrive/tiles/tile_41.png"
noise_example = r"/content/drive/MyDrive/tiles/tile_54.png"
geo_example = r"/content/drive/MyDrive/tiles/tile_55.png"


good_img = cv2.imread(good_example)
blur_img = cv2.imread(blur_example)
noise_img = cv2.imread(noise_example)
geo_img = cv2.imread(geo_example)


good_blur = checker.blur_score(good_img)
good_noise = checker.noise_score(good_img)
good_geo = checker.geo_score(good_img)

blur_ref = checker.blur_score(blur_img)
noise_ref = checker.noise_score(noise_img)
geo_ref = checker.geo_score(geo_img)


# -----------------------------
# Traverse Folder
# -----------------------------

folder_path = r"/content/drive/MyDrive/tiles"

results = []

for file in os.listdir(folder_path):

    if file.endswith(".png"):

        path = os.path.join(folder_path,file)

        img = cv2.imread(path)

        if img is None:
            print("Skipping corrupted image:",file)
            continue

        blur = checker.blur_score(img)
        noise = checker.noise_score(img)
        geo = checker.geo_score(img)
        align_error = checker.alignment_error(img, good_img)

        defect_score = (
            abs(blur - good_blur)/max(good_blur,1) +
            abs(noise - good_noise)/max(good_noise,1) +
            abs(geo - good_geo)/max(good_geo,1) +
            align_error/25
        )

        results.append((file,blur,noise,geo,align_error,defect_score))


# -----------------------------
# Sort Images by Defect Score
# -----------------------------

results.sort(key=lambda x: x[5], reverse=True)

bad_images = results[:20]
good_images = results[20:]


print("\nDetected Bad Images:\n")

for img in bad_images:

    name, blur, noise, geo, align_error, score = img

    defects = []

    # Slightly reduced transform sensitivity
    if align_error > 25 or geo >= geo_ref*0.9:
        defects.append("Transform")

    # blur detection
    if blur <= blur_ref:
        defects.append("Blur")

    # noise detection
    if noise >= noise_ref:
        defects.append("Noise")

    if len(defects) == 0:
        defects.append("Unknown")

    print(name,"--> Bad Image ->",", ".join(defects))


print("\nDetected Good Images:\n")

for img in good_images:

    print(img[0],"--> Good Image")


print("\nTotal Bad Images:",len(bad_images))
print("Total Good Images:",len(good_images))


Detected Bad Images:

tile_60.png --> Bad Image -> Transform
tile_59.png --> Bad Image -> Transform, Noise
tile_50.png --> Bad Image -> Transform, Noise
tile_51.png --> Bad Image -> Transform
tile_54.png --> Bad Image -> Transform, Noise
tile_47.png --> Bad Image -> Transform
tile_45.png --> Bad Image -> Transform, Blur
tile_55.png --> Bad Image -> Transform, Blur
tile_42.png --> Bad Image -> Transform, Blur
tile_52.png --> Bad Image -> Transform, Blur
tile_46.png --> Bad Image -> Transform, Noise
tile_56.png --> Bad Image -> Transform
tile_53.png --> Bad Image -> Transform
tile_44.png --> Bad Image -> Transform, Noise
tile_41.png --> Bad Image -> Transform, Blur
tile_49.png --> Bad Image -> Transform, Noise
tile_24.png --> Bad Image -> Transform
tile_57.png --> Bad Image -> Transform, Noise
tile_04.png --> Bad Image -> Transform, Noise
tile_43.png --> Bad Image -> Transform, Noise

Detected Good Images:

tile_05.png --> Good Image
tile_31.png --> Good Image
tile_29.png --> Good Image